In [1]:
import torch
import sys, os, pdb
import torch.nn as nn
import argparse, logging
import torch.nn.functional as F

import librosa

from pathlib import Path

from src.model.voice_quality.wavlm_voice_quality import WavLMWrapper

/root/miniconda3/envs/api-2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/vox/src/model/voice_quality/wavlm_voice_quality.py:5: UserWarning: Module 'speechbrain.lobes.models.huggingface_transformers' was deprecated, redirecting to 'speechbrain.integrations.huggingface'. Please update your script.
  from speechbrain.lobes.models.huggingface_transformers.huggingface import make_padding_masks


In [2]:
labels = [
    'shrill', 'nasal', 'deep',  # Pitch
    'silky', 'husky', 'raspy', 'guttural', 'vocal-fry', # Texture
    'booming', 'authoritative', 'loud', 'hushed', 'soft', # Volume
    'crisp', 'slurred', 'lisp', 'stammering', # Clarity
    'singsong', 'pitchy', 'flowing', 'monotone', 'staccato', 'punctuated', 'enunciated',  'hesitant', # Rhythm
]

In [3]:
device = "cuda"

# Define the model
# Note that ensemble yields the better performance than the single model
wavlm_model = WavLMWrapper.from_pretrained("tiantiaf/wavlm-large-voice-quality").to(device).eval()

In [4]:
# audio sample frequency is set to 16kHz
data, _ = librosa.load("./misc/LJ037-0171.wav", sr=16000)
data = torch.tensor(data)[None].to(device)
wavlm_logits = wavlm_model(data, return_feature=False)
wavlm_prob = nn.Sigmoid()(torch.tensor(wavlm_logits))

# In practice, a larger threshold would remove some noise, but it is best to aggregate predictions per speaker
threshold = 0.5
predictions = (wavlm_prob > threshold).int().detach().cpu().numpy()[0].tolist()
print(wavlm_prob.shape)

torch.Size([1, 25])


/root/miniconda3/envs/api-2/lib/python3.11/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/tmp/ipykernel_9870/3420166297.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavlm_prob = nn.Sigmoid()(torch.tensor(wavlm_logits))


In [8]:
for i, label in enumerate(labels):
    print("{}: {:.3f}".format(label, wavlm_prob[0][i].item()))

shrill: 0.000
nasal: 0.000
deep: 0.000
silky: 0.150
husky: 0.001
raspy: 0.000
guttural: 0.000
vocal-fry: 0.000
booming: 0.000
authoritative: 0.495
loud: 0.389
hushed: 0.000
soft: 0.577
crisp: 0.980
slurred: 0.000
lisp: 0.000
stammering: 0.000
singsong: 0.000
pitchy: 0.000
flowing: 0.307
monotone: 0.000
staccato: 0.000
punctuated: 0.000
enunciated: 0.005
hesitant: 0.000


In [10]:
wavlm_prob

tensor([[4.2508e-06, 6.8357e-07, 3.9953e-05, 1.5043e-01, 1.0861e-03, 1.3505e-05,
         5.2767e-25, 3.6760e-12, 4.5979e-04, 4.9462e-01, 3.8863e-01, 0.0000e+00,
         5.7732e-01, 9.7998e-01, 5.1855e-07, 0.0000e+00, 1.5683e-11, 2.5314e-09,
         2.4330e-09, 3.0734e-01, 9.4698e-34, 0.0000e+00, 1.1709e-07, 4.7588e-03,
         6.0104e-05]])